In [4]:
def flatten_posts_dict(posts_by_tag):
    flat_posts = {}
    for tag, posts in posts_by_tag.items():
        for uri, post in posts.items():
            flat_posts[uri] = post
    return flat_posts

In [17]:
def add_reposters_to_posts(posts_dict, users):
    for post in posts_dict.values():
        post["reposted_by"] = []

    for did, user in users.items():
        for uri in user.get("reposted_posts", []):
            if uri in posts_dict:
                posts_dict[uri]["reposted_by"].append(did)

    for post in posts_dict.values():
        post["repost_user_unobservable"] = (
            post.get("repostCount", 0) > 0 and not post["reposted_by"]
        )


In [ ]:
import json

with open("TEST_X2.json", "r", encoding="utf-8") as f:
    raw_user_data = json.load(f)

with open("posts.json", "r", encoding="utf-8") as f:
    posts = json.load(f)


In [18]:
post_dict = flatten_posts_dict(posts)
add_reposters_to_posts(post_dict, raw_user_data)

In [30]:
def print_basic_stats(users, posts_dict):
    # -------------------------
    # Authors followed per user
    # -------------------------
    follow_counts = [
        len(u.get("follows_authors", []))
        for u in users.values()
    ]

    avg_authors_followed = (
        sum(follow_counts) / len(follow_counts)
        if follow_counts else 0
    )
    max_authors_followed = max(follow_counts) if follow_counts else 0
    # -------------------------
    # Reposts per post
    # -------------------------
    repost_counts = [
        len(p.get("reposted_by", []))
        for p in posts_dict.values()
        if "reposted_by" in p
    ]
    pos_instances = sum(repost_counts)

    avg_reposts_per_post = (
        sum(repost_counts) / len(repost_counts)
        if repost_counts else 0
    )
    max_reposts_per_post = max(repost_counts) if repost_counts else 0

    posts_with_reposts = sum(c > 0 for c in repost_counts)
    pct_posts_with_reposts = (
        100 * posts_with_reposts / len(post_dict)
        if repost_counts else 0
    )

    # -------------------------
    # Reposts per user (from reposter_dict output)
    # -------------------------
    reposts_per_user = [
        len(u.get("reposted_posts", []))
        for u in users.values()
    ]
    pos_instances_2 = sum(reposts_per_user)

    users_with_reposts = sum(c > 0 for c in reposts_per_user)
    pct_users_with_reposts = (
        100 * users_with_reposts / len(reposts_per_user)
        if reposts_per_user else 0
    )

    # -------------------------
    # Repost timestamps (from history)
    # -------------------------
    total_reposts = 0
    reposts_with_time = 0

    for u in users.values():
        for h in u.get("history", []):
            if h.get("activity_type") == "repost":
                total_reposts += 1
                if h.get("reposted_at"):
                    reposts_with_time += 1

    pct_reposts_with_time = (
        100 * reposts_with_time / total_reposts
        if total_reposts else 0
    )

    # -------------------------
    # Print results
    # -------------------------
    print("===== BASIC DATASET STATS =====")
    print(f"Users: {len(users)}")
    print(f"Posts: {len(posts_dict)}")
    print()

    print(
        f"Authors followed per user → "
        f"avg: {avg_authors_followed:.2f}, "
        f"max: {max_authors_followed}"
    )

    print(
        f"Reposts per post → "
        f"avg: {avg_reposts_per_post:.2f}, "
        f"max: {max_reposts_per_post}"
    )

    print(
        f"Posts with ≥1 repost → "
        f"{posts_with_reposts} "
        f"({pct_posts_with_reposts:.2f}%)"
    )

    print(
        f"Users with ≥1 repost → "
        f"{users_with_reposts} "
        f"({pct_users_with_reposts:.2f}%)"
    )

    print(
        f"Reposts with timestamp → "
        f"{reposts_with_time}/{total_reposts} "
        f"({pct_reposts_with_time:.2f}%)"
    )

    print(
        f"Positive instances: "
        f"{pos_instances} and "
        f"{pos_instances_2}"
        )


In [7]:
def print_posts_with_more_than_x_reposts(posts_dict,x):
    count = sum(
        p.get("repostCount", 0) > x
        for p in posts_dict.values()
    )
    print(f"Posts with >{x} reposts: {count}/{len(posts_dict)}")


print_posts_with_more_than_x_reposts(post_dict,0)

Posts with >0 reposts: 42625/169904


In [31]:
print_basic_stats(raw_user_data,post_dict)

===== BASIC DATASET STATS =====
Users: 74383
Posts: 169904

Authors followed per user → avg: 5.24, max: 68
Reposts per post → avg: 0.87, max: 100
Posts with ≥1 repost → 36336 (21.39%)
Users with ≥1 repost → 50850 (68.36%)
Reposts with timestamp → 2285844/2285844 (100.00%)
Positive instances: 147786 and 147786


In [27]:
def count_unique_authors(posts_dict):
    """
    Returns the number of unique author DIDs in posts_dict.
    """
    return len({
        (
            p.get("author", {}).get("did")
            or p.get("post", {}).get("author", {}).get("did")
        )
        for p in posts_dict.values()
        if p
    })
num_authors = count_unique_authors(post_dict)
print(f"Unique authors: {num_authors}")

Unique authors: 29746
